# Notebook 4a — Seeing recovery families, one step at a time

**A small, real-data companion to Notebook 3.** We keep Tacloban City, Palo, Guiuan,
Alangalang, Ormoc City and Baybay City. Start with their existing observations;
inspect what gets summarised; then try two exploratory groups and put them on a map.

Run in the **same environment and project folder as Notebook 3**. The setup below
copies its data access and RQ functions. Only the output folder changes. This notebook
builds four-day municipality profiles; it omits the daily, POI and NGCP analyses.
If adding these cells beneath an already-run Notebook 3, start at **“1. Keep the actual
four-day maps”**: all required source objects and functions already exist there.

| Step | What you will see | Question to ask |
|---|---|---|
| 1 | Original six curves; baseline, impact and later maps | What is actually being measured? |
| 2 | Baseline brightness versus variability; two real pixel examples | Can a small denominator amplify variation? |
| 3 | Four-day observations → 20-day summaries → a feature matrix | What exactly enters grouping? |
| 4 | Ordinary curves, pointwise band and functional boxplot side by side | What changes when we rank entire curves? |
| 5 | Pairwise distances, two families and their municipality map | Which behaviours look similar, and where? |
| 6 | T50/T80/T90 values, statuses and a separate milestone grouping | Does timing tell the same story? |
| 7 | Brighter-support rerun, maps and assignment changes | Is the conclusion sensitive to dim pixels? |
| 8 | A pre-event reference split, with held-out observations | How stable is the apparent baseline before Haiyan? |

**Continuity with Notebook 3:** MQF = 0; GHSL G3 (22, 23, 30); daily P95 clipping
over the same six municipalities; four-day **medians**; pixel medians over the
60-day baseline; ≥3 baseline composites per pixel; positive baseline; SC ≥10%.
Recovery remains `100 × sum(current radiance) / sum(matched pixel baselines)`.
The numerator and denominator use the same currently observed pixels.

This file contains runnable plot cells, not precomputed findings: the uploaded
Notebook 3 has no saved outputs, and its local Zarr and geospatial data were not
attached. No substitute observations are generated.

## Setup from Notebook 3
These are the original imports, paths, data preparation and metric functions.
The source cell index is recorded above each cell. No clustering choices occur here.
The small POC starts immediately after this setup. Additional dependencies are
`scipy` and `scikit-learn`; install them in the same environment if absent.

In [1]:
# Copied from Notebook 3, source cell index 2
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import json
import re
import unicodedata
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import Point

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Copied from Notebook 3, source cell index 3
# ============================================================
# 2. PATHS AND ANALYTICAL SETTINGS
# ============================================================

PROJECT_DIR_OVERRIDE = Path(
    "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery"
)

project_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    PROJECT_DIR_OVERRIDE,
]
PROJECT_DIR = next(
    (candidate for candidate in project_candidates if (candidate / "datasets").exists()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project datasets directory. "
        "Set PROJECT_DIR_OVERRIDE to the project root."
    )

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
NGCP_CSV_PATH = DATA_DIR / "ngcp" / "NGCP_Hourly_Demand.csv"


def find_dataset(patterns, label):
    '''Return the first unique match and show all candidates.'''

    matches = []

    for pattern in patterns:
        matches.extend(DATA_DIR.glob(pattern))

    matches = sorted({path.resolve() for path in matches})

    if not matches:
        raise FileNotFoundError(
            f"No {label} matched under {DATA_DIR}.\n"
            f"Patterns: {patterns}"
        )

    if len(matches) > 1:
        print(f"{label}: multiple matches; using {matches[0]}")
        for candidate in matches:
            print("  ", candidate)

    return matches[0]


GHSL_PATH = find_dataset(
    [
        "VNP46/GHSL_SMOD_E2015.tif",
        "ghsl/GHSL_SMOD_E2015.tif",
        "**/*GHSL*SMOD*.tif",
    ],
    "GHSL SMOD raster",
)

MUNICITIES_PATH = find_dataset(
    [
        "**/*MuniCities*.shp",
        "**/*municities*.shp",
        "**/*Muni*Cit*.shp",
        "**/*Municipal*.shp",
    ],
    "MuniCities shapefile",
)

ROADS_PATH = find_dataset(
    [
        "**/*Roads*.shp",
        "**/*roads*.shp",
        "**/*Road*.shp",
    ],
    "Roads shapefile",
)

HAIYAN_TRACK_PATH = find_dataset(
    [
        "yolanda-path-line-/Yolanda Path Line.shp",
        "**/Yolanda Path Line.shp",
        "**/*Yolanda*Path*.shp",
        "**/*Haiyan*Path*.shp",
    ],
    "Haiyan/Yolanda path shapefile",
)

OUTPUT_DIR = PROJECT_DIR / "output" / "visual_recovery_poc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Event and temporal design
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS = 60
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
PROFILE_END = EVENT_DATE + pd.Timedelta(days=363)
BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)

STAGE_WINDOWS = {
    "Baseline": (BASELINE_START, PRE_EVENT_END),
    "Stage 1 (0–59 d)": (
        EVENT_DATE,
        EVENT_DATE + pd.Timedelta(days=59),
    ),
    "Stage 2 (60–119 d)": (
        EVENT_DATE + pd.Timedelta(days=60),
        EVENT_DATE + pd.Timedelta(days=119),
    ),
    "Stage 3 (120–179 d)": (
        EVENT_DATE + pd.Timedelta(days=120),
        EVENT_DATE + pd.Timedelta(days=179),
    ),
    "Extended (180–363 d)": (
        EVENT_DATE + pd.Timedelta(days=180),
        PROFILE_END,
    ),
}

# VNP46A2 layers
DNB_BAND = "DNB_BRDF_Corrected_NTL"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"
MQF_BAND = "Mandatory_Quality_Flag"
SPATIAL_DIMS = ("y", "x")

# RQ1-informed settlement configurations. G3 is fixed for this notebook.
GHSL_MASKS = {
    "G2": (23, 30),
    "G3": (22, 23, 30),
    "G4": (21, 22, 23, 30),
}
SETTLEMENT_MASK = "G3"
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0

MIN_BASELINE_OBSERVATIONS = {
    1: 5,
    4: 3,
}

# Metric admissibility and persistence rules. T50/T80/T90 are measured
# against the matched 60-day baseline after the observed post-event nadir.
PERSISTENCE_BLOCKS = 2
MIN_EVENT_RETENTION_PCT = 20.0
MIN_EVENT_COMPOSITES = 8
MAX_INTERPRETABLE_GAP_DAYS = 24

# Fixed six-location analytical design. These are affected comparisons,
# not unaffected controls.
CORE_LOCATIONS = [
    {
        "recovery_setting": "Regional urban impact centre",
        "core_location": "Tacloban",
        "municipality_name": "Tacloban City",
        "poi_name": "Tacloban City Centre",
        "why": (
            "Major urban/service centre within the severely affected "
            "Leyte Gulf corridor"
        ),
    },
    {
        "recovery_setting": "Adjacent impacted settlement",
        "core_location": "Palo",
        "municipality_name": "Palo",
        "poi_name": "Palo Centre",
        "why": (
            "Tests recovery close to Tacloban but under a different "
            "settlement and reconstruction context"
        ),
    },
    {
        "recovery_setting": "First-landfall setting",
        "core_location": "Guiuan",
        "municipality_name": "Guiuan",
        "poi_name": "Guiuan Centre",
        "why": (
            "Represents the initial Haiyan landfall and severe eastern exposure"
        ),
    },
    {
        "recovery_setting": "Inland contrast",
        "core_location": "Alangalang",
        "municipality_name": "Alangalang",
        "poi_name": "Alangalang Centre",
        "why": (
            "Separates inland/peri-urban recovery from coastal and "
            "storm-surge trajectories"
        ),
    },
    {
        "recovery_setting": "Western urban comparison",
        "core_location": "Ormoc",
        "municipality_name": "Ormoc City",
        "poi_name": "Ormoc City Centre",
        "why": (
            "Tests whether recovery differs outside the principal eastern "
            "impact corridor"
        ),
    },
    {
        "recovery_setting": "Western secondary city",
        "core_location": "Baybay",
        "municipality_name": "Baybay City",
        "poi_name": "Baybay City Centre",
        "why": (
            "Adds a second western urban comparison under a different local context"
        ),
    },
]

TARGET_UNITS = [location["municipality_name"] for location in CORE_LOCATIONS]
LOCATION_ORDER = TARGET_UNITS.copy()
POI_ORDER = [location["poi_name"] for location in CORE_LOCATIONS]
DISPLAY_NAMES = {
    location["municipality_name"]: location["core_location"]
    for location in CORE_LOCATIONS
}
LOCATION_COLORS = {
    "Tacloban City": "#0072B2",
    "Palo": "#E69F00",
    "Guiuan": "#009E73",
    "Alangalang": "#CC79A7",
    "Ormoc City": "#D55E00",
    "Baybay City": "#56B4E9",
}

# Every local stage map uses the same square extent around its POI.
LOCAL_MAP_RADIUS_PIXELS = 14

FOCUS_UNIT = "Tacloban City"
POI_KERNEL_SIZE = 5

KNOWN_FILL_VALUES = (-9999.0, -32768.0, 6553.5, 65535.0)

print("Project:", PROJECT_DIR)
print("MuniCities:", MUNICITIES_PATH)
print("Roads:", ROADS_PATH)
print("Haiyan/Yolanda path:", HAIYAN_TRACK_PATH)
print("Primary GHSL mask:", SETTLEMENT_MASK, GHSL_MASKS[SETTLEMENT_MASK])
print("Spatial-completeness gate:", f"{SPATIAL_COMPLETENESS_PCT:.0f}%")

Project: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery
MuniCities: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/boundaries/MuniCities/MuniCities.shp
Roads: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/boundaries/Roads/roads.shp
Haiyan/Yolanda path: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/yolanda-path-line-/Yolanda Path Line.shp
Primary GHSL mask: G3 (22, 23, 30)
Spatial-completeness gate: 10%


In [3]:
# Copied from Notebook 3, source cell index 5
# ============================================================
# 4. VNP46A2 HELPERS
# ============================================================


def open_zarr_safely(path):
    try:
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks="auto",
            mask_and_scale=True,
            decode_cf=True,
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks=None,
            mask_and_scale=True,
            decode_cf=True,
        )


def standardise_date_dimension(ds):
    if "date" not in ds.variables:
        raise KeyError(f"No `date` variable found. Variables: {list(ds.variables)}")

    if "date" not in ds.coords:
        ds = ds.set_coords("date")

    observation_dim = ds["date"].dims[0]
    dates = pd.DatetimeIndex(pd.to_datetime(ds["date"].values)).normalize()
    ds = ds.assign_coords(date=(observation_dim, dates.values))

    if observation_dim != "date":
        ds = ds.swap_dims({observation_dim: "date"})

    return ds.sortby("date")


def prepare_spatial_metadata(ds):
    ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

    if ds.rio.crs is None and "spatial_ref" in ds.variables:
        attrs = ds["spatial_ref"].attrs
        stored_crs = attrs.get("crs_wkt") or attrs.get("spatial_ref")

        if stored_crs is not None:
            ds = ds.rio.write_crs(stored_crs, inplace=False)

    if ds.rio.crs is None:
        x_min = float(ds["x"].min())
        x_max = float(ds["x"].max())
        y_min = float(ds["y"].min())
        y_max = float(ds["y"].max())

        if (
            -180 <= x_min <= 180
            and -180 <= x_max <= 180
            and -90 <= y_min <= 90
            and -90 <= y_max <= 90
        ):
            ds = ds.rio.write_crs("EPSG:4326", inplace=False)
        else:
            raise ValueError("The VNP46A2 CRS could not be recovered.")

    return ds


def clean_radiance(values):
    cleaned = values.astype("float32").where(np.isfinite(values))
    fill_values = list(KNOWN_FILL_VALUES)

    for source in (values.attrs, values.encoding):
        for key in ("_FillValue", "missing_value"):
            if source.get(key) is not None:
                fill_values.append(source[key])

    for fill_value in fill_values:
        try:
            fill_value = float(fill_value)
            if np.isfinite(fill_value):
                cleaned = cleaned.where(~np.isclose(cleaned, fill_value))
        except (TypeError, ValueError):
            continue

    return cleaned.where(cleaned >= 0)


if not A2_ZARR_PATH.exists():
    raise FileNotFoundError(f"VNP46A2 Zarr not found:\n{A2_ZARR_PATH}")

a2 = prepare_spatial_metadata(
    standardise_date_dimension(open_zarr_safely(A2_ZARR_PATH))
).sel(date=slice(ANALYSIS_START, PROFILE_END))

required_bands = [DNB_BAND, MQF_BAND]
missing_bands = [band for band in required_bands if band not in a2.data_vars]

if missing_bands:
    raise KeyError(
        f"Missing A2 bands: {missing_bands}\n"
        f"Available: {list(a2.data_vars)}"
    )

dnb = clean_radiance(a2[DNB_BAND])
mqf = a2[MQF_BAND]
dnb, mqf = xr.align(dnb, mqf, join="inner")

gap_filled = (
    clean_radiance(a2[GAP_FILLED_BAND])
    if GAP_FILLED_BAND in a2.data_vars
    else None
)

print("A2 dimensions:", dict(a2.sizes))
print("A2 CRS:", a2.rio.crs)
print("Available dates:", pd.Timestamp(a2.date.min().item()).date(), "to", pd.Timestamp(a2.date.max().item()).date())

A2 dimensions: {'date': 544, 'y': 674, 'x': 473}
A2 CRS: EPSG:4326
Available dates: 2013-05-12 to 2014-11-06


In [4]:
# Copied from Notebook 3, source cell index 6
# ============================================================
# 5. LOAD MUNICIPALITIES AND ROADS; DISCOVER FIELD NAMES
# ============================================================


def normalise_column_name(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def resolve_column(gdf, candidates, label):
    lookup = {normalise_column_name(column): column for column in gdf.columns}

    for candidate in candidates:
        key = normalise_column_name(candidate)
        if key in lookup:
            return lookup[key]

    for candidate in candidates:
        key = normalise_column_name(candidate)
        for normalised, column in lookup.items():
            if key in normalised or normalised in key:
                return column

    raise KeyError(
        f"Could not identify the {label} field. "
        f"Available columns: {list(gdf.columns)}"
    )


def canonical_unit_name(value):
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(character for character in text if not unicodedata.combining(character))
    tokens = re.sub(r"[^A-Z0-9]+", " ", text.upper()).split()
    stop_words = {"CITY", "OF", "MUNICIPALITY", "MUNICIPAL", "MUN"}
    return " ".join(token for token in tokens if token not in stop_words)


municipalities = gpd.read_file(MUNICITIES_PATH)

if municipalities.crs is None:
    raise ValueError("MuniCities shapefile does not contain a CRS.")

municipalities = municipalities.loc[
    municipalities.geometry.notna() & ~municipalities.geometry.is_empty
].copy()

try:
    municipalities["geometry"] = municipalities.geometry.make_valid()
except AttributeError:
    municipalities["geometry"] = municipalities.geometry.buffer(0)

NAME_COLUMN = resolve_column(
    municipalities,
    [
        "ADM3_EN",
        "ADM3_NAME",
        "MuniCity",
        "Muni_City",
        "CITY_MUN",
        "NAME_2",
        "NAME_3",
        "LGU_NAME",
        "MUNICIPALITY",
        "NAME",
    ],
    "city/municipality name",
)

province_candidates = [
    "ADM2_EN",
    "ADM2_NAME",
    "PROVINCE",
    "PROV_NAME",
    "NAME_1",
]

try:
    PROVINCE_COLUMN = resolve_column(
        municipalities,
        province_candidates,
        "province name",
    )
except KeyError:
    PROVINCE_COLUMN = None

municipalities["unit_name"] = municipalities[NAME_COLUMN].astype(str).str.strip()
municipalities["unit_key"] = municipalities["unit_name"].map(canonical_unit_name)
municipalities["province_name"] = (
    municipalities[PROVINCE_COLUMN].astype(str).str.strip()
    if PROVINCE_COLUMN is not None
    else "Not supplied"
)

# Exact canonical matching first; conservative substring matching second.
selected_rows = []
unmatched_targets = []

for target in TARGET_UNITS:
    target_key = canonical_unit_name(target)
    exact = municipalities.loc[municipalities["unit_key"] == target_key]

    if len(exact) == 1:
        selected_rows.append(exact.index[0])
        continue

    partial = municipalities.loc[
        municipalities["unit_key"].str.contains(target_key, regex=False)
        | pd.Series(
            [target_key in key for key in municipalities["unit_key"]],
            index=municipalities.index,
        )
    ]

    if len(partial) == 1:
        selected_rows.append(partial.index[0])
    else:
        unmatched_targets.append(target)

selected_municipalities = municipalities.loc[
    list(dict.fromkeys(selected_rows))
].copy()

target_name_lookup = {
    canonical_unit_name(name): name
    for name in TARGET_UNITS
}
selected_municipalities["source_unit_name"] = selected_municipalities["unit_name"]
selected_municipalities["unit_name"] = selected_municipalities["unit_key"].map(
    target_name_lookup
)

if selected_municipalities["unit_name"].isna().any():
    raise ValueError("A selected boundary could not be assigned to the fixed six-location design.")

if selected_municipalities.empty:
    raise ValueError(
        "None of TARGET_UNITS matched the MuniCities shapefile. "
        "Inspect the available names printed below and edit TARGET_UNITS."
    )

selected_municipalities["profile_id"] = np.arange(
    1,
    len(selected_municipalities) + 1,
)

roads = gpd.read_file(ROADS_PATH)

if roads.crs is None:
    raise ValueError("Roads shapefile does not contain a CRS.")

roads = roads.loc[roads.geometry.notna() & ~roads.geometry.is_empty].copy()

print("Name field:", NAME_COLUMN)
print("Province field:", PROVINCE_COLUMN)
print("Matched targets:")
display(selected_municipalities[["unit_name", "province_name", "unit_key"]])

if unmatched_targets:
    print("Unmatched targets; edit TARGET_UNITS if needed:", unmatched_targets)

print("First 30 available municipality names:")
display(
    municipalities[["unit_name", "province_name"]]
    .sort_values(["province_name", "unit_name"])
    .head(30)
)

haiyan_track = gpd.read_file(HAIYAN_TRACK_PATH)
if haiyan_track.crs is None:
    raise ValueError("The Haiyan/Yolanda path shapefile does not contain a CRS.")
haiyan_track = haiyan_track.loc[
    haiyan_track.geometry.notna() & ~haiyan_track.geometry.is_empty
].copy()

Name field: NAME_2
Province field: PROVINCE
Matched targets:


,unit_name,province_name,unit_key
955,Tacloban City,Leyte,TACLOBAN
926,Palo,Leyte,PALO
620,Guiuan,Eastern Samar,GUIUAN
897,Alangalang,Leyte,ALANGALANG
925,Ormoc City,Leyte,ORMOC
902,Baybay City,Leyte,BAYBAY


First 30 available municipality names:


,unit_name,province_name
106,Bangued,Abra
107,Boliney,Abra
108,Bucay,Abra
109,Bucloc,Abra
110,Daguioman,Abra
111,Danglas,Abra
112,Dolores,Abra
113,La Paz,Abra
114,Lacub,Abra
115,Lagangilang,Abra


In [5]:
# Copied from Notebook 3, source cell index 7
# ============================================================
# 6. ALIGN GHSL AND RASTERIZE MUNICIPAL SUPPORT
# ============================================================

ghsl = rxr.open_rasterio(GHSL_PATH, masked=True)

if "band" in ghsl.dims:
    ghsl = ghsl.isel(band=0, drop=True)

if ghsl.rio.crs is None:
    raise ValueError("The GHSL raster does not contain a CRS.")

viirs_template = dnb.isel(date=0, drop=True)

ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template,
    resampling=Resampling.nearest,
).assign_coords(x=viirs_template["x"], y=viirs_template["y"])

ghsl_mask = ghsl_viirs.isin(GHSL_MASKS[SETTLEMENT_MASK]).fillna(False)

municipalities_raster_crs = municipalities.to_crs(a2.rio.crs)
selected_raster_crs = selected_municipalities.to_crs(a2.rio.crs)
roads_raster_crs = roads.to_crs(a2.rio.crs)

raster_bounds = dnb.rio.bounds()
municipalities_raster_crs = municipalities_raster_crs.cx[
    raster_bounds[0]:raster_bounds[2],
    raster_bounds[1]:raster_bounds[3],
].copy()


def rasterize_units(gdf, value_column):
    shapes = [
        (geometry, int(value))
        for geometry, value in zip(gdf.geometry, gdf[value_column])
        if geometry is not None and not geometry.is_empty
    ]

    values = rasterize(
        shapes,
        out_shape=(dnb.sizes["y"], dnb.sizes["x"]),
        transform=dnb.rio.transform(recalc=True),
        fill=0,
        all_touched=False,
        dtype="int32",
    )

    return xr.DataArray(
        values,
        dims=SPATIAL_DIMS,
        coords={"y": dnb["y"], "x": dnb["x"]},
    )


municipalities_raster_crs = municipalities_raster_crs.copy()
municipalities_raster_crs["all_unit_id"] = np.arange(
    1,
    len(municipalities_raster_crs) + 1,
)

all_zone_id = rasterize_units(municipalities_raster_crs, "all_unit_id")
selected_zone_id = rasterize_units(selected_raster_crs, "profile_id")

study_mask = selected_zone_id > 0
rq_base_mask = (ghsl_mask & study_mask).compute()

print("GHSL classes:", GHSL_MASKS[SETTLEMENT_MASK])
print("RQ settlement pixels within the six selected municipalities:", f"{int(rq_base_mask.sum().item()):,}")
print("Any selected LGU represented on the VIIRS grid:", bool((selected_zone_id > 0).any().item()))

haiyan_track_raster_crs = haiyan_track.to_crs(a2.rio.crs)

GHSL classes: (22, 23, 30)
RQ settlement pixels within the six selected municipalities: 616
Any selected LGU represented on the VIIRS grid: True


In [6]:
# Copied from Notebook 3, source cell index 8
# ============================================================
# 7. BUILD THE RELIABILITY-QUALIFIED CUBE
# ============================================================

rq_unclipped = dnb.where((mqf == 0) & rq_base_mask)
rq_quantile_source = rq_unclipped

if hasattr(rq_unclipped.data, "rechunk"):
    spatial_axes = {
        rq_unclipped.get_axis_num(dimension): -1
        for dimension in SPATIAL_DIMS
    }
    rq_quantile_source = rq_unclipped.copy(
        data=rq_unclipped.data.rechunk(spatial_axes)
    )

rq_daily_p95 = (
    rq_quantile_source
    .quantile(RQ_CLIP_PERCENTILE / 100.0, dim=SPATIAL_DIMS, skipna=True)
    .squeeze(drop=True)
    .compute()
)

rq_cube = xr.where(
    rq_unclipped > rq_daily_p95,
    rq_daily_p95,
    rq_unclipped,
)
rq_cube.name = "reliability_qualified_ntl"

print(
    "Daily P95 range:",
    f"{float(rq_daily_p95.min(skipna=True)):.2f}",
    "to",
    f"{float(rq_daily_p95.max(skipna=True)):.2f}",
    "nW cm⁻² sr⁻¹",
)

/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


Daily P95 range: 0.01 to 19.16 nW cm⁻² sr⁻¹


In [7]:
# Copied from Notebook 3, source cell index 9
# ============================================================
# 8. PROFILE FUNCTIONS
# ============================================================


def crop_to_support(cube, support_mask):
    support_values = np.asarray(support_mask.fillna(False).values, dtype=bool)
    rows, columns = np.where(support_values)

    if len(rows) == 0:
        return None, None

    y_slice = slice(rows.min(), rows.max() + 1)
    x_slice = slice(columns.min(), columns.max() + 1)

    return (
        cube.isel(y=y_slice, x=x_slice),
        support_mask.isel(y=y_slice, x=x_slice),
    )


def build_pixel_matched_profile(
    cube,
    support_mask,
    aggregation_days,
    unit_name,
    unit_type,
    method="Reliability-qualified DNB-BRDF",
):
    '''Build a daily or non-overlapping multi-day profile.'''

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(support_mask)
    )

    dates = pd.DatetimeIndex(selected["date"].values).normalize()
    block_numbers = np.floor_divide(
        (dates - EVENT_DATE).days,
        aggregation_days,
    ).astype(int)

    composites = (
        selected
        .assign_coords(block=("date", block_numbers))
        .groupby("block")
        .median(dim="date", skipna=True)
    )

    blocks = composites["block"].values.astype(int)
    block_start = EVENT_DATE + pd.to_timedelta(blocks * aggregation_days, unit="D")
    block_end = block_start + pd.Timedelta(days=aggregation_days - 1)

    baseline_blocks = blocks[
        (block_start >= BASELINE_START)
        & (block_end <= PRE_EVENT_END)
    ]

    if len(baseline_blocks) == 0:
        raise ValueError(f"{unit_name}: no complete baseline blocks were found.")

    expected_baseline_blocks = BASELINE_DAYS // aggregation_days
    if BASELINE_DAYS % aggregation_days != 0:
        raise ValueError(
            "BASELINE_DAYS must be divisible by aggregation_days so the "
            "baseline contains complete, non-overlapping composites."
        )
    if len(baseline_blocks) != expected_baseline_blocks:
        raise ValueError(
            f"{unit_name}: expected {expected_baseline_blocks} complete baseline "
            f"blocks inside {BASELINE_START.date()}–{PRE_EVENT_END.date()}, "
            f"but found {len(baseline_blocks)}."
        )

    baseline_composites = composites.sel(block=baseline_blocks).compute()
    baseline_observations = baseline_composites.notnull().sum(dim="block")

    ntl0 = baseline_composites.median(dim="block", skipna=True)
    baseline_quantiles = baseline_composites.quantile(
        [0.25, 0.75],
        dim="block",
        skipna=True,
    )
    ntl0_q25 = baseline_quantiles.sel(quantile=0.25, drop=True)
    ntl0_q75 = baseline_quantiles.sel(quantile=0.75, drop=True)

    minimum_baseline = MIN_BASELINE_OBSERVATIONS[aggregation_days]
    fixed_mask = (
        support_mask
        & (baseline_observations >= minimum_baseline)
        & np.isfinite(ntl0)
        & (ntl0 > 0)
    ).compute()

    fixed_pixel_count = int(fixed_mask.sum().item())

    if fixed_pixel_count == 0:
        raise ValueError(
            f"{unit_name}: no baseline-lit {SETTLEMENT_MASK} pixels "
            f"met the baseline requirement."
        )

    paired_valid = composites.notnull() & fixed_mask & ntl0.notnull()
    valid_pixel_count = paired_valid.sum(dim=SPATIAL_DIMS)
    spatial_coverage_pct = 100.0 * valid_pixel_count / fixed_pixel_count

    current_radiance = composites.where(paired_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    matched_baseline = ntl0.where(paired_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    recovery_pct = 100.0 * current_radiance / matched_baseline

    # Raw magnitude before baseline normalization: spatial median across the
    # currently valid, fixed baseline-lit pixels.
    raw_median_source = composites.where(paired_valid)
    matched_median_source = ntl0.where(paired_valid)

    if hasattr(raw_median_source.data, "rechunk"):
        raw_spatial_axes = {
            raw_median_source.get_axis_num(dimension): -1
            for dimension in SPATIAL_DIMS
        }
        raw_median_source = raw_median_source.copy(
            data=raw_median_source.data.rechunk(raw_spatial_axes)
        )
    if hasattr(matched_median_source.data, "rechunk"):
        matched_spatial_axes = {
            matched_median_source.get_axis_num(dimension): -1
            for dimension in SPATIAL_DIMS
        }
        matched_median_source = matched_median_source.copy(
            data=matched_median_source.data.rechunk(matched_spatial_axes)
        )

    raw_median_ntl = raw_median_source.median(
        dim=SPATIAL_DIMS,
        skipna=True,
    )
    matched_baseline_median_ntl = matched_median_source.median(
        dim=SPATIAL_DIMS,
        skipna=True,
    )

    uncertainty_valid = (
        paired_valid
        & ntl0_q25.notnull()
        & ntl0_q75.notnull()
        & (ntl0_q25 > 0)
    )
    uncertainty_current = composites.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    baseline_q25_sum = ntl0_q25.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    baseline_q75_sum = ntl0_q75.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )

    recovery_low_pct = 100.0 * uncertainty_current / baseline_q75_sum
    recovery_high_pct = 100.0 * uncertainty_current / baseline_q25_sum

    reduced = xr.Dataset(
        {
            "current_radiance": current_radiance,
            "matched_baseline_radiance": matched_baseline,
            "raw_median_ntl": raw_median_ntl,
            "matched_baseline_median_ntl": matched_baseline_median_ntl,
            "recovery_pct": recovery_pct,
            "recovery_low_pct": recovery_low_pct,
            "recovery_high_pct": recovery_high_pct,
            "spatial_coverage_pct": spatial_coverage_pct,
            "valid_pixel_count": valid_pixel_count,
        }
    ).compute()

    profile = (
        reduced
        .to_dataframe()
        .reset_index()
        .sort_values("block")
        .reset_index(drop=True)
    )
    profile["date_start"] = EVENT_DATE + pd.to_timedelta(
        profile["block"] * aggregation_days,
        unit="D",
    )
    profile["date_end"] = profile["date_start"] + pd.Timedelta(
        days=aggregation_days - 1
    )
    profile["date_mid"] = profile["date_start"] + pd.to_timedelta(
        (aggregation_days - 1) / 2,
        unit="D",
    )

    below_gate = profile["spatial_coverage_pct"] < SPATIAL_COMPLETENESS_PCT
    profile.loc[
        below_gate,
        [
            "current_radiance",
            "matched_baseline_radiance",
            "raw_median_ntl",
            "matched_baseline_median_ntl",
            "recovery_pct",
            "recovery_low_pct",
            "recovery_high_pct",
        ],
    ] = np.nan

    profile["observation_status"] = np.where(
        profile["recovery_pct"].notna(),
        "observed",
        "not observable",
    )
    profile["unit_name"] = unit_name
    profile["unit_type"] = unit_type
    profile["method"] = method
    profile["aggregation_days"] = aggregation_days
    profile["ghsl_mask"] = SETTLEMENT_MASK
    profile["sc_threshold_pct"] = SPATIAL_COMPLETENESS_PCT
    profile["baseline_start"] = BASELINE_START
    profile["baseline_end"] = PRE_EVENT_END
    profile["baseline_days"] = BASELINE_DAYS
    profile["temporal_composite_statistic"] = "median"
    profile["raw_spatial_statistic"] = "median"
    profile["baseline_definition"] = (
        "per-pixel median of complete median reliability-qualified composites "
        "within the 60 days before Haiyan"
    )

    baseline_rows = profile.loc[
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END)
    ]

    report = {
        "unit_name": unit_name,
        "unit_type": unit_type,
        "aggregation_days": aggregation_days,
        "ghsl_mask": SETTLEMENT_MASK,
        "ghsl_classes": str(GHSL_MASKS[SETTLEMENT_MASK]),
        "sc_threshold_pct": SPATIAL_COMPLETENESS_PCT,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "baseline_blocks": len(baseline_blocks),
        "minimum_baseline_observations": minimum_baseline,
        "temporal_composite_statistic": "median",
        "raw_spatial_statistic": "median",
        "fixed_baseline_pixels": fixed_pixel_count,
        "median_baseline_coverage_pct": baseline_rows[
            "spatial_coverage_pct"
        ].median(),
    }

    return profile, composites.where(fixed_mask), ntl0.where(fixed_mask), fixed_mask, report

In [8]:
# Copied from Notebook 3, source cell index 13
# ============================================================
# 12. RECOVERY METRIC FUNCTIONS
# ============================================================


def expected_event_blocks(aggregation_days=4):
    inclusive_days = (PROFILE_END - EVENT_DATE).days + 1
    return int(np.ceil(inclusive_days / aggregation_days))


def longest_missing_run_days(profile, aggregation_days=4):
    expected_blocks = np.arange(
        0,
        int(np.floor((PROFILE_END - EVENT_DATE).days / aggregation_days)) + 1,
    )
    observed_blocks = set(
        profile.loc[
            (profile["date_start"] >= EVENT_DATE)
            & (profile["date_start"] <= PROFILE_END)
            & profile["recovery_pct"].notna(),
            "block",
        ].astype(int)
    )
    missing = np.array(
        [block not in observed_blocks for block in expected_blocks],
        dtype=int,
    )

    longest = current = 0
    for value in missing:
        current = current + 1 if value else 0
        longest = max(longest, current)

    return longest * aggregation_days


def observability_class(profile, aggregation_days=4):
    post = profile.loc[
        (profile["date_start"] >= EVENT_DATE)
        & (profile["date_start"] <= PROFILE_END)
    ]
    retained = int(post["recovery_pct"].notna().sum())
    expected = expected_event_blocks(aggregation_days)
    retained_pct = 100.0 * retained / expected if expected else np.nan
    max_gap = longest_missing_run_days(profile, aggregation_days)

    if (
        retained >= MIN_EVENT_COMPOSITES
        and retained_pct >= MIN_EVENT_RETENTION_PCT
        and max_gap <= MAX_INTERPRETABLE_GAP_DAYS
    ):
        label, flag = "interpretable with interval timing", "OBS_OK"
    elif retained >= MIN_EVENT_COMPOSITES and retained_pct >= MIN_EVENT_RETENTION_PCT:
        label, flag = "observation-limited", "OBS_LIMITED"
    else:
        label, flag = "not observable", "NOT_OBSERVABLE"

    return {
        "retained_composites": retained,
        "expected_composites": expected,
        "retained_pct": retained_pct,
        "max_gap_days": max_gap,
        "observability": label,
        "quality_flag": flag,
    }


def persistent_crossing(
    profile,
    threshold,
    value_column="recovery_pct",
    start_block=0,
):
    """First persistent threshold return at or after the observed impact block."""

    post = (
        profile.loc[
            (profile["date_start"] >= EVENT_DATE)
            & (profile["date_start"] <= PROFILE_END)
            & (profile["block"] >= start_block)
            & profile[value_column].notna()
        ]
        .sort_values("block")
        .set_index("block", drop=False)
    )

    for block, row in post.iterrows():
        required = list(range(int(block), int(block) + PERSISTENCE_BLOCKS))
        if not set(required).issubset(post.index):
            continue
        if not (post.loc[required, value_column] >= threshold).all():
            continue

        previous_below = post.loc[
            (post["block"] < block) & (post[value_column] < threshold)
        ]
        lower_day = int((row["date_start"] - EVENT_DATE).days)
        if not previous_below.empty:
            lower_day = max(
                0,
                int(
                    (
                        previous_below.iloc[-1]["date_end"]
                        + pd.Timedelta(days=1)
                        - EVENT_DATE
                    ).days
                ),
            )

        upper_day = int((row["date_start"] - EVENT_DATE).days)
        return {
            "day": upper_day,
            "lower_day": lower_day,
            "upper_day": upper_day,
            "date": row["date_start"],
            "observed_value": float(row[value_column]),
        }

    return None


def format_interval(crossing):
    if crossing is None:
        return None
    return f"{crossing['lower_day']}–{crossing['upper_day']} d"


def format_baseline_range(optimistic, conservative):
    if optimistic is None and conservative is None:
        return None
    earliest = optimistic["day"] if optimistic is not None else None
    latest = conservative["day"] if conservative is not None else None
    if earliest is not None and latest is not None:
        return f"{min(earliest, latest)}–{max(earliest, latest)} d"
    if earliest is not None:
        return f"≥{earliest} d; conservative crossing absent"
    return f"≤{latest} d; optimistic crossing absent"


def calculate_recovery_metrics(profile):
    """Calculate impact and persistent return-to-baseline milestones.

    T50/T80/T90 mean the first of two consecutive admissible four-day
    composites at or above 50/80/90% of the matched 60-day baseline,
    searched only after the observed Stage-1 nadir. They are not fractions
    of the shock-to-baseline amplitude.
    """

    profile = profile.sort_values("date_start").copy()
    obs = observability_class(profile, aggregation_days=4)
    event = profile.loc[
        (profile["date_start"] >= EVENT_DATE)
        & (profile["date_start"] <= PROFILE_END)
    ]
    impact_window = event.loc[
        event["date_start"] <= EVENT_DATE + pd.Timedelta(days=59)
    ].dropna(subset=["recovery_pct"])

    impact_row = None
    if impact_window.empty:
        impact_recovery = impact_drop = np.nan
        impact_drop_low = impact_drop_high = np.nan
        impact_date, impact_block = pd.NaT, np.nan
    else:
        impact_row = impact_window.loc[impact_window["recovery_pct"].idxmin()]
        impact_recovery = float(impact_row["recovery_pct"])
        impact_date = impact_row["date_start"]
        impact_block = int(impact_row["block"])
        impact_drop = 100.0 - impact_recovery
        impact_drop_low = 100.0 - float(impact_row["recovery_high_pct"])
        impact_drop_high = 100.0 - float(impact_row["recovery_low_pct"])

    crossings = {}
    for threshold in (50, 80, 90):
        threshold_lost = bool(
            impact_row is not None and impact_recovery < threshold
        )

        if threshold_lost:
            central = persistent_crossing(
                profile,
                threshold,
                "recovery_pct",
                start_block=impact_block,
            )
            optimistic_lost = (
                pd.notna(impact_row["recovery_high_pct"])
                and float(impact_row["recovery_high_pct"]) < threshold
            )
            conservative_lost = (
                pd.notna(impact_row["recovery_low_pct"])
                and float(impact_row["recovery_low_pct"]) < threshold
            )
            optimistic = (
                persistent_crossing(
                    profile,
                    threshold,
                    "recovery_high_pct",
                    start_block=impact_block,
                )
                if optimistic_lost
                else None
            )
            conservative = (
                persistent_crossing(
                    profile,
                    threshold,
                    "recovery_low_pct",
                    start_block=impact_block,
                )
                if conservative_lost
                else None
            )
        else:
            central = optimistic = conservative = None

        if impact_row is None:
            status = "not observable in the 0–59 day impact window"
        elif not threshold_lost:
            status = "threshold not lost at observed nadir"
        elif central is not None:
            status = "supported; persistent return after observed nadir"
        elif obs["quality_flag"] == "NOT_OBSERVABLE":
            status = "not observable"
        elif obs["quality_flag"] == "OBS_LIMITED":
            status = "not identifiable: observation-limited"
        else:
            status = f"not recovered by {(PROFILE_END - EVENT_DATE).days} d"

        crossings[threshold] = {
            "central": central,
            "optimistic": optimistic,
            "conservative": conservative,
            "lost": threshold_lost,
            "status": status,
        }

    post_observed = event.dropna(subset=["recovery_pct"]).copy()
    slope = np.nan
    if not post_observed.empty and pd.notna(impact_date):
        slope_end_day = (
            crossings[90]["central"]["day"]
            if crossings[90]["central"] is not None
            else int((PROFILE_END - EVENT_DATE).days)
        )
        slope_data = post_observed.loc[
            (post_observed["date_start"] >= impact_date)
            & (
                post_observed["date_start"]
                <= EVENT_DATE + pd.Timedelta(days=slope_end_day)
            )
        ]
        if len(slope_data) >= 3:
            x = (slope_data["date_start"] - EVENT_DATE).dt.days.to_numpy(dtype=float)
            y = slope_data["recovery_pct"].to_numpy(dtype=float)
            slope = float(np.polyfit(x, y, 1)[0])

    stage3 = event.loc[
        (event["date_start"] >= EVENT_DATE + pd.Timedelta(days=120))
        & (event["date_start"] <= EVENT_DATE + pd.Timedelta(days=179))
    ].dropna(subset=["recovery_pct"])
    stability_mad = (
        float(np.median(np.abs(stage3["recovery_pct"] - stage3["recovery_pct"].median())))
        if not stage3.empty
        else np.nan
    )
    stable_share = (
        float(stage3["recovery_pct"].between(90, 110).sum() * 100.0 / len(stage3))
        if not stage3.empty
        else np.nan
    )

    result = {
        "unit_name": profile["unit_name"].iloc[0],
        "unit_type": profile["unit_type"].iloc[0],
        "metric_definition": (
            "persistent return to 50/80/90% of the matched 60-day baseline "
            "after the observed 0–59 day nadir"
        ),
        "ghsl_mask": SETTLEMENT_MASK,
        "ghsl_classes": str(GHSL_MASKS[SETTLEMENT_MASK]),
        "sc_threshold_pct": SPATIAL_COMPLETENESS_PCT,
        "baseline_days": BASELINE_DAYS,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "impact_date": impact_date,
        "impact_block": impact_block,
        "impact_recovery_pct": impact_recovery,
        "impact_drop_pct": impact_drop,
        "impact_drop_range_pct": (
            f"{impact_drop_low:.1f}–{impact_drop_high:.1f}"
            if np.isfinite(impact_drop_low) and np.isfinite(impact_drop_high)
            else None
        ),
        "recovery_slope_pct_per_day": slope,
        "observed_days_below_baseline": int(
            (post_observed["recovery_pct"] < 100).sum() * 4
        ),
        "stage3_stability_mad_pct": stability_mad,
        "stage3_within_90_110_pct": stable_share,
        "median_event_sc_pct": float(event["spatial_coverage_pct"].median()),
        "minimum_event_sc_pct": float(event["spatial_coverage_pct"].min()),
        **obs,
    }

    for threshold, crossing in crossings.items():
        result[f"T{threshold}_threshold_lost"] = crossing["lost"]
        result[f"T{threshold}_day"] = (
            crossing["central"]["day"]
            if crossing["central"] is not None
            else np.nan
        )
        result[f"T{threshold}_observation_interval"] = format_interval(
            crossing["central"]
        )
        result[f"T{threshold}_baseline_range"] = format_baseline_range(
            crossing["optimistic"],
            crossing["conservative"],
        )
        result[f"T{threshold}_status"] = crossing["status"]

    return result

## 1. Keep the actual four-day maps

This calls **the same `build_pixel_matched_profile` function** as Notebook 3.
We keep its returned map cube and baseline image, instead of discarding them.
There is one small dictionary entry per municipality, not a new data pipeline.

First compare the ordinary time series. The raw panel is Notebook 3's spatial
median; the recovery panel is its matched **ratio of sums**, so dividing the raw
median by another median would not reproduce it. Missing four-day blocks remain gaps.

In [9]:
from scipy.stats import rankdata
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

POC_UNIT = "Tacloban City"  # Change to one of LOCATION_ORDER for another mapped example.
BIN_DAYS = 20              # Five original four-day blocks per bin; no time warping.
MIN_BIN_COMPOSITES = 2     # Exploratory choice: show counts, do not fill missing bins.
POC_K = 2                 # A visual demonstration, not an estimated number of families.
DIM_QUANTILE = 0.25        # Sensitivity only: remove the lowest baseline quartile per LGU.
SAVE_HTML = False
POC_OUT = PROJECT_DIR / "output" / "visual_recovery_poc"
POC_OUT.mkdir(parents=True, exist_ok=True)
figures = {}

def show(fig, name, title, height=470):
    fig.update_layout(template="plotly_white", title=dict(text=title, x=0.02),
        height=height, font=dict(family="Arial", size=13, color="#243B5A"),
        paper_bgcolor="rgba(0,0,0,0)", margin=dict(l=65, r=35, t=100, b=65),
        legend=dict(orientation="h", y=1.14, x=0), hovermode="closest")
    figures[name] = fig
    if SAVE_HTML:
        fig.write_html(POC_OUT / f"{name}.html", include_plotlyjs=True)
    fig.show()

def full_series(profile, field, blocks=range(-15, 91)):
    return profile.set_index("block")[field].reindex(blocks)

def trace(x, y, name, color, **kw):
    return go.Scatter(x=x, y=y, name=name, mode="lines+markers",
        line=dict(color=color), marker=dict(size=4), connectgaps=False, **kw)

cache, profiles = {}, []
for row in selected_raster_crs.itertuples():
    support = (selected_zone_id == int(row.profile_id)) & ghsl_mask
    local, support = crop_to_support(rq_cube, support)
    if local is None:
        raise ValueError(f"No source support for {row.unit_name}; inspect Notebook 3 setup.")
    p, C, B, fixed, report = build_pixel_matched_profile(
        local, support, 4, row.unit_name, "City/municipality")
    cache[row.unit_name] = dict(C=C.compute(), B=B.compute(), fixed=fixed,
        support=support, report=report)
    profiles.append(p)
poc_profiles = pd.concat(profiles, ignore_index=True)
metrics = pd.DataFrame([calculate_recovery_metrics(p) for p in profiles]).set_index("unit_name")
display(metrics[["impact_recovery_pct", "retained_pct", "max_gap_days", "quality_flag"]])

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Original raw magnitude", "Original matched-baseline recovery"))
for name in LOCATION_ORDER:
    p = poc_profiles.loc[poc_profiles.unit_name == name]
    for col, field in enumerate(("raw_median_ntl", "recovery_pct"), 1):
        s = full_series(p, field)
        fig.add_trace(trace(4*s.index + 1.5, s.values, name, LOCATION_COLORS[name],
            legendgroup=name, showlegend=(col == 1)), row=1, col=col)
    fig.update_xaxes(title_text="Days since Haiyan")
fig.add_hline(y=100, line_dash="dot", row=1, col=2)
fig.add_vline(x=0, line_dash="dash", line_color="#0057FF")
fig.update_yaxes(title_text="Spatial median (nW cm⁻² sr⁻¹)", row=1, col=1)
fig.update_yaxes(title_text="% of matched baseline", row=1, col=2)
show(fig, "01_original_curves", "1 · The same six municipality profiles as Notebook 3")

/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_i

,impact_recovery_pct,retained_pct,max_gap_days,quality_flag
unit_name,,,,
Tacloban City,6.223722,76.923077,16,OBS_OK
Palo,12.865019,78.021978,16,OBS_OK
Guiuan,8.978182,71.428571,20,OBS_OK
Alangalang,5.131015,80.219780,16,OBS_OK
Ormoc City,13.551199,81.318681,12,OBS_OK
Baybay City,14.696360,76.923077,16,OBS_OK


**One municipality, three actual maps.** The first is its pixel baseline `B`.
The other two show each pixel's median `100 × C/B` during Stage 1 and Stage 3,
displayed only with ≥3 observed composites in that phase. This extra display gate
does not change the municipality time series. Pixel ratios explain spatial variation;
their median is not the municipality ratio of sums. Blank pixels mean unsupported,
not zero light. The two recovery maps share one scale; values above 200% saturate
visually but remain in the data and hover values.

In [10]:
def add_outline(fig, name, row=1, col=1):
    geom = selected_raster_crs.loc[selected_raster_crs.unit_name == name].geometry.iloc[0]
    polygons = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for polygon in polygons:
        for ring in [polygon.exterior, *polygon.interiors]:
            x, y = ring.xy
            fig.add_trace(go.Scatter(x=list(x), y=list(y), mode="lines",
                line=dict(color="#303030", width=0.8), showlegend=False,
                hoverinfo="skip"), row=row, col=col)
    # Preserve map proportions in the source raster CRS.
    axis = col  # Raster map examples in this POC occupy a single row.
    xref = "x" if axis == 1 else f"x{axis}"
    ratio = (1 / np.cos(np.deg2rad(geom.centroid.y))
             if selected_raster_crs.crs.is_geographic else 1)
    fig.update_yaxes(scaleanchor=xref, scaleratio=ratio, row=row, col=col)
    fig.update_xaxes(showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, row=row, col=col)

example = cache[POC_UNIT]
C, B = example["C"], example["B"]
R = 100 * C / B
phase_maps = []
for lo, hi in [(0, 14), (30, 44)]:
    phase = R.reindex(block=np.arange(lo, hi+1))
    phase_maps.append(phase.median("block", skipna=True).where(phase.count("block") >= 3))
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.10,
    subplot_titles=("60-day pixel baseline", "Stage 1: days 0–59", "Stage 3: days 120–179"))
for col, m in enumerate([B, *phase_maps], 1):
    fig.add_trace(go.Heatmap(x=m.x.values, y=m.y.values, z=m.values,
        coloraxis="coloraxis" if col == 1 else "coloraxis2", hoverongaps=False,
        hovertemplate="x=%{x}<br>y=%{y}<br>Value=%{z:.2f}<extra></extra>"), row=1, col=col)
    add_outline(fig, POC_UNIT, col=col)
fig.update_layout(coloraxis=dict(colorscale="Inferno", cmin=0,
    colorbar=dict(title="Radiance", x=0.27, len=0.7, thickness=12)),
    coloraxis2=dict(colorscale="Viridis", cmin=0, cmax=200,
    colorbar=dict(title="% baseline", x=1.02, len=0.7, thickness=12)))
show(fig, "02_actual_pixel_maps", f"1 · {POC_UNIT}: baseline → impact → later radiance", 540)

## 2. Does a dim baseline amplify variability?

For every retained baseline pixel, calculate its pre-event median `B` and robust
relative variability `1.4826 × MAD / B` across the 15 four-day baseline slots.
The count table matters: three observations permit a baseline in Notebook 3, but
give a weak variability estimate. The scatter uses logarithmic axes for positive
values; the count of zero-MAD pixels is reported separately.

Then show two **actual pixels**, chosen near the 10th and 90th baseline-brightness
percentiles within the example municipality. Their identities are determined from
pre-event brightness, not from their post-event behaviour. Compare raw radiance
with the same observations expressed as percentages. Two examples illustrate the
denominator issue; they cannot establish that all dim pixels are unreliable.

In [11]:
pixel_tables = []
for name, obj in cache.items():
    base = obj["C"].reindex(block=np.arange(-15, 0))
    b = obj["B"]
    mad = abs(base - b).median("block", skipna=True)
    table = xr.Dataset(dict(B=b, robust_cv=1.4826*mad/b,
        baseline_n=base.count("block"))).to_dataframe().reset_index()
    table = table.loc[np.isfinite(table.B) & (table.B > 0)].copy()
    table["unit_name"] = name
    pixel_tables.append(table)
pixels = pd.concat(pixel_tables, ignore_index=True)
display(pixels.groupby("unit_name").agg(pixels=("B", "size"),
    median_B=("B", "median"), median_baseline_n=("baseline_n", "median"),
    median_robust_cv=("robust_cv", "median")))
positive = pixels.loc[pixels.robust_cv > 0]
print("Zero-MAD pixels omitted from log scatter:", int((pixels.robust_cv == 0).sum()))
plot_pixels = positive.sample(min(6000, len(positive)), random_state=42)
fig = px.scatter(plot_pixels, x="B", y="robust_cv", color="unit_name",
    color_discrete_map=LOCATION_COLORS, log_x=True, log_y=True, opacity=0.4,
    hover_data=["x", "y", "baseline_n"])
fig.update_xaxes(title="Pixel baseline radiance")
fig.update_yaxes(title="Pre-event robust relative variability")
show(fig, "03_baseline_variability", "2 · Brightness and variability, before looking at recovery")

local_pixels = pixels.loc[pixels.unit_name == POC_UNIT].copy()
chosen = []
for q in (0.10, 0.90):
    target = local_pixels.B.quantile(q)
    chosen.append(local_pixels.loc[(local_pixels.B-target).abs().idxmin()])
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "Example locations", "Raw four-day radiance", "The same pixels ÷ their baseline"))
fig.add_trace(go.Heatmap(x=B.x.values, y=B.y.values, z=B.values,
    colorscale="Greys", showscale=False, hoverongaps=False), row=1, col=1)
add_outline(fig, POC_UNIT)
for p, label, color in zip(chosen, ("Dim example", "Bright example"), ("#D55E00", "#0072B2")):
    values = C.sel(x=p.x, y=p.y).reindex(block=np.arange(-15, 45))
    days = values.block.values * 4 + 1.5
    fig.add_trace(go.Scatter(x=[p.x], y=[p.y], text=[label], mode="markers",
        marker=dict(size=11, color=color, line=dict(color="white", width=1)),
        name=label, legendgroup=label), row=1, col=1)
    for col, y in ((2, values.values), (3, 100*values.values/p.B)):
        fig.add_trace(trace(days, y, label, color, showlegend=False,
            legendgroup=label), row=1, col=col)
        fig.add_vline(x=0, line_dash="dash", line_color="#0057FF", row=1, col=col)
        fig.update_xaxes(title_text="Days since Haiyan", row=1, col=col)
fig.add_hline(y=100, line_dash="dot", row=1, col=3)
fig.update_yaxes(title_text="nW cm⁻² sr⁻¹", row=1, col=2)
fig.update_yaxes(title_text="% of pixel baseline", row=1, col=3)
show(fig, "04_two_real_pixels", f"2 · {POC_UNIT}: follow two real pixels through normalization", 500)
display(pd.DataFrame(chosen)[["x", "y", "B", "baseline_n", "robust_cv"]])

,pixels,median_B,median_baseline_n,median_robust_cv
unit_name,,,,
Alangalang,42,0.366093,10.0,0.522474
Baybay City,90,0.440727,7.0,0.414854
Guiuan,29,1.022667,5.0,0.433942
Ormoc City,200,0.704548,7.0,0.414510
Palo,102,0.992972,9.0,0.476222
Tacloban City,147,5.251113,7.0,0.362874


Zero-MAD pixels omitted from log scatter: 0


,x,y,B,baseline_n,robust_cv
22,124.972916,11.239583,0.931214,7,0.521118
55,124.993750,11.243750,11.178396,6,0.253582


## 3. Show exactly what will enter grouping

Group the first **180 post-event days** using nine fixed, event-aligned 20-day bins.
Each value is the median of at least two supported four-day municipality observations
out of five possible slots. This is a visible coarse summary, not another recovery
estimate or a replacement for Notebook 3. It can hide short outages; keep the original
curve next to it. T50/T80/T90 will still use the original four-day data through day 363.

The matrix shows values and the second panel shows counts. A municipality enters the
functional boxplot and curve clustering only if all nine bins are available.
No missing bin is interpolated, replaced with zero, or treated as recovery.

In [12]:
def make_features(frame):
    post = frame.loc[frame.block.between(0, 44)].copy()
    post["bin"] = (post.block.astype(int) * 4 // BIN_DAYS).astype(int)
    g = post.groupby(["unit_name", "bin"], observed=True).recovery_pct
    med = g.median().unstack("bin").reindex(index=LOCATION_ORDER, columns=range(9))
    count = g.count().unstack("bin").reindex(index=LOCATION_ORDER, columns=range(9)).fillna(0)
    return med.where(count >= MIN_BIN_COMPOSITES), count

features, counts = make_features(poc_profiles)
bin_days = np.arange(9) * BIN_DAYS + (BIN_DAYS-1)/2
bin_labels = [f"{i*20}–{i*20+19}" for i in range(9)]
eligible = features.notna().all(axis=1)
display(pd.DataFrame({"complete_bins": features.notna().sum(axis=1),
    "eligible_for_curve_group": eligible}))
fig = go.Figure()
p = poc_profiles.loc[poc_profiles.unit_name == POC_UNIT]
s = full_series(p, "recovery_pct", range(45))
fig.add_trace(trace(s.index*4+1.5, s.values, "Original four-day observations", "#999999"))
fig.add_trace(trace(bin_days, features.loc[POC_UNIT], "20-day medians", "#0072B2"))
fig.add_hline(y=100, line_dash="dot")
fig.update_xaxes(title="Days since Haiyan")
fig.update_yaxes(title="% of matched baseline")
show(fig, "05_observations_to_bins", f"3 · {POC_UNIT}: observations → grouping features")

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Recovery features (% baseline)", "Retained four-day observations (out of 5)"),
    horizontal_spacing=0.23)
for col, mat in ((1, features), (2, counts)):
    fig.add_trace(go.Heatmap(x=bin_labels, y=LOCATION_ORDER, z=mat.values,
        text=mat.round(0).values, texttemplate="%{text}", hoverongaps=False,
        colorscale="Viridis", zmin=0, zmax=200 if col == 1 else 5,
        showscale=False), row=1, col=col)
    fig.update_yaxes(autorange="reversed", row=1, col=col)
    fig.update_xaxes(title_text="Post-event day range", row=1, col=col)
fig.update_layout(plot_bgcolor="#E5E5E5")
show(fig, "06_feature_matrix_and_counts", "3 · Every input value, and how much evidence supports it", 450)

,complete_bins,eligible_for_curve_group
unit_name,,
Tacloban City,8,False
Palo,8,False
Guiuan,8,False
Alangalang,8,False
Ormoc City,9,True
Baybay City,9,True


## 4. Ordinary time series versus a functional boxplot

All three panels use the **same complete 20-day curves**, so the only change is the
summary. Ordinary curves retain every municipality's identity. A pointwise band
computes quartiles independently at each time bin. A functional boxplot ranks whole
curves by **modified band depth**: how often a curve lies inside the bands formed by
pairs of curves across time. The deepest observed curve is the functional median.

The central envelope contains the deepest `ceil(n/2)` whole curves. Its boundaries
are not pointwise quartiles, and it is not a confidence interval. We also show the
outer non-outlier envelope and curves outside the conventional 1.5× central-envelope
fence. With at most six curves, these are descriptive flags, not evidence of error;
flagged curves remain in the clustering. Time bins receive equal weight.

In [13]:
def functional_summary(values):
    values = np.asarray(values, float)
    n = len(values)
    if n < 3 or not np.isfinite(values).all():
        raise ValueError("Need at least three complete curves on the same time grid.")
    below = rankdata(values, method="min", axis=0) - 1
    above = n - rankdata(values, method="max", axis=0)
    depth = (1 - (below*(below-1) + above*(above-1))/(n*(n-1))).mean(axis=1)
    order = np.argsort(-depth, kind="stable")  # Ties follow displayed municipality order.
    centre = values[order[:int(np.ceil(n/2))]]
    low, high = centre.min(axis=0), centre.max(axis=0)
    outlier = ((values < low-1.5*(high-low)) | (values > high+1.5*(high-low))).any(axis=1)
    return dict(depth=depth, order=order, low=low, high=high, outlier=outlier,
        median=values[order[0]], outer_low=values[~outlier].min(axis=0),
        outer_high=values[~outlier].max(axis=0))

def band(fig, x, low, high, name, color, col):
    fig.add_trace(go.Scatter(x=x, y=low, mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip"), row=1, col=col)
    fig.add_trace(go.Scatter(x=x, y=high, mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor=color, name=name), row=1, col=col)

complete = features.loc[eligible]
if len(complete) < 3:
    print("Functional boxplot withheld: fewer than three complete curves. Inspect the count matrix.")
else:
    values = complete.to_numpy(float)
    fb = functional_summary(values)
    fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=(
        "Ordinary time series", "Pointwise median + IQR", "Functional boxplot"))
    for name, vals in complete.iterrows():
        fig.add_trace(trace(bin_days, vals, name, LOCATION_COLORS[name]), row=1, col=1)
    q25, med, q75 = np.quantile(values, [.25, .5, .75], axis=0)
    band(fig, bin_days, q25, q75, "Pointwise IQR", "rgba(0,114,178,.25)", 2)
    fig.add_trace(trace(bin_days, med, "Pointwise median", "#0072B2"), row=1, col=2)
    band(fig, bin_days, fb["outer_low"], fb["outer_high"], "Outer envelope", "rgba(0,0,0,.08)", 3)
    band(fig, bin_days, fb["low"], fb["high"], "Deepest 50% envelope", "rgba(0,158,115,.3)", 3)
    deepest = complete.index[fb["order"][0]]
    fig.add_trace(trace(bin_days, fb["median"], f"Functional median: {deepest}", "#006644"), row=1, col=3)
    for i in np.where(fb["outlier"])[0]:
        fig.add_trace(trace(bin_days, values[i], f"Flag: {complete.index[i]}", "#D55E00"), row=1, col=3)
    fig.add_hline(y=100, line_dash="dot")
    fig.update_xaxes(title_text="Days since Haiyan")
    fig.update_yaxes(title_text="% of matched baseline", row=1, col=1)
    show(fig, "07_functional_vs_pointwise", "4 · Same curves; three ways to display them", 580)
    display(pd.DataFrame({"band_depth": fb["depth"],
        "central_half": np.isin(np.arange(len(values)), fb["order"][:int(np.ceil(len(values)/2))]),
        "descriptive_outlier_flag": fb["outlier"]}, index=complete.index))

Functional boxplot withheld: fewer than three complete curves. Inspect the count matrix.


## 5. From similarity to two mapped families

Distance is the root-mean-square difference between municipalities' recovery
percentages across the nine shared time bins. **Lower distance means more similar
levels and timing.** K-means uses the same Euclidean geometry (RMS differs only by
a common scale). We keep the common calendar and do not standardise each curve to
zero mean: that would discard sustained losses or above-baseline levels.

This is **temporal grouping followed by spatial mapping**, not spatially constrained
clustering. Coordinates are deliberately absent from the feature matrix. Two nearby
municipalities can enter different groups; distant municipalities can behave similarly.
With six selected municipalities, two groups are a teaching choice, not a regional
typology. Singleton groups are highlighted for inspection.

In [14]:
GROUP_COLORS = {"Family 1": "#0072B2", "Family 2": "#D55E00", "Not grouped": "#BBBBBB"}
def group_curves(matrix):
    good = matrix.dropna()
    labels = pd.Series("Not grouped", index=matrix.index, dtype=object)
    if len(good) < 4 or len(np.unique(good.to_numpy(float), axis=0)) < POC_K:
        return labels
    model = KMeans(n_clusters=POC_K, n_init=30, random_state=42).fit(good.to_numpy(float))
    # Deterministic naming: Family 1 has the lower mean feature value.
    order = np.argsort(model.cluster_centers_.mean(axis=1))
    names = {int(k): f"Family {i+1}" for i, k in enumerate(order)}
    labels.loc[good.index] = [names[int(k)] for k in model.labels_]
    return labels

curve_groups = group_curves(features)
if len(complete):
    distance = pd.DataFrame(pairwise_distances(complete)/np.sqrt(complete.shape[1]),
        index=complete.index, columns=complete.index)
    fig = px.imshow(distance, text_auto=".1f", color_continuous_scale="Blues",
        labels=dict(color="RMS difference (pp)"))
    show(fig, "08_pairwise_similarity", "5 · Which recovery curves are closest?", 530)

def family_map(groups, name, title):
    geo = selected_municipalities.to_crs(4326).copy()
    geo["family"] = geo.unit_name.map(groups).fillna("Not grouped")
    geo["map_id"] = geo.unit_name
    # IDs join directly to actual source polygons; labels are not geocoded guesses.
    geojson = json.loads(geo[["map_id", "geometry"]].set_index("map_id").to_json())
    fig = px.choropleth(geo, geojson=geojson, locations="map_id", color="family",
        color_discrete_map=GROUP_COLORS, hover_name="unit_name",
        category_orders={"family": list(GROUP_COLORS)})
    fig.update_traces(marker_line_color="#333333", marker_line_width=0.8)
    pts = geo.to_crs(32651).representative_point().to_crs(4326)
    fig.add_trace(go.Scattergeo(lon=pts.x, lat=pts.y, text=geo.unit_name,
        mode="text", textfont=dict(size=11), showlegend=False))
    fig.update_geos(fitbounds="locations", visible=False, projection_type="mercator")
    show(fig, name, title, 600)

fig = go.Figure()
for name, vals in complete.iterrows():
    family = curve_groups.loc[name]
    fig.add_trace(trace(bin_days, vals, f"{name} · {family}", GROUP_COLORS[family]))
for family in ("Family 1", "Family 2"):
    members = complete.loc[curve_groups.reindex(complete.index) == family]
    if len(members):
        fig.add_trace(go.Scatter(x=bin_days, y=members.mean(axis=0), mode="lines",
            name=f"{family} mean (n={len(members)})",
            line=dict(color=GROUP_COLORS[family], width=6, dash="dash")))
fig.add_hline(y=100, line_dash="dot")
fig.update_xaxes(title="Days since Haiyan")
fig.update_yaxes(title="% of matched baseline")
show(fig, "09_two_curve_families", "5 · See the members before interpreting the family means", 550)
family_map(curve_groups, "10_mapped_curve_families", "5 · Where the temporally similar municipalities are")
display(curve_groups.rename("curve_family").to_frame())
print("Group sizes:", curve_groups.value_counts().to_dict())
if (curve_groups.value_counts().drop("Not grouped", errors="ignore") == 1).any():
    print("A family has one member: inspect that case; do not generalise it as a stable class.")

,curve_family
unit_name,
Tacloban City,Not grouped
Palo,Not grouped
Guiuan,Not grouped
Alangalang,Not grouped
Ormoc City,Not grouped
Baybay City,Not grouped


Group sizes: {'Not grouped': 6}


## 6. Grouping by T50, T80 and T90 is a different question

Keep Notebook 3's definition: the first of two consecutive admissible four-day
composites above the threshold, **after the observed Stage-1 nadir**, through day 363.
These are returns to percentages of baseline, not percentages of impact repaired.

Show the original status and uncertainty fields beside each time. A threshold not
lost has no return time; a missing crossing is not day zero or day 363. Timing-only
clustering uses municipalities with all three supported times and `OBS_OK`.
If fewer than four qualify, it is withheld and the status table remains the result.
The interval fields are Notebook 3's observation brackets and baseline sensitivity
ranges, not confidence intervals or exact physical restoration dates.

The milestone groups use equal-weight differences in days. They can disagree with
curve groups because they discard trajectory details and cover a longer window
(363 versus 179 days). Family numbers in the two analyses are separate labels.

In [15]:
milestone_rows = []
for name in LOCATION_ORDER:
    m = metrics.loc[name]
    for t in (50, 80, 90):
        milestone_rows.append(dict(unit_name=name, threshold=f"T{t}", day=m[f"T{t}_day"],
            status=m[f"T{t}_status"], observation_bracket=m[f"T{t}_observation_interval"],
            baseline_sensitivity=m[f"T{t}_baseline_range"], quality_flag=m.quality_flag))
milestones = pd.DataFrame(milestone_rows)
display(milestones)
fig = px.scatter(milestones.dropna(subset=["day"]), x="day", y="unit_name",
    color="threshold", symbol="threshold", hover_data=["status", "observation_bracket",
    "baseline_sensitivity", "quality_flag"], category_orders={"unit_name": LOCATION_ORDER})
fig.update_traces(marker=dict(size=12))
fig.update_xaxes(title="Supported return day since Haiyan (original four-day profiles)")
fig.update_yaxes(title=None)
show(fig, "11_original_recovery_milestones", "6 · T50 / T80 / T90: hover for timing and baseline uncertainty")

tcols = [f"T{t}_day" for t in (50, 80, 90)]
t_features = metrics[tcols].reindex(LOCATION_ORDER).copy()
t_ok = metrics.reindex(LOCATION_ORDER).quality_flag.eq("OBS_OK")
for t in (50, 80, 90):
    t_ok &= metrics.reindex(LOCATION_ORDER)[f"T{t}_status"].str.startswith("supported", na=False)
t_features.loc[~t_ok, :] = np.nan
time_groups = group_curves(t_features)
group_comparison = pd.DataFrame(dict(curve_family=curve_groups,
    milestone_family=time_groups, milestone_eligible=t_ok))
display(group_comparison.join(metrics[tcols + ["quality_flag"]]))
if time_groups.ne("Not grouped").any():
    family_map(time_groups, "12_mapped_milestone_families", "6 · Separate families based only on T50 / T80 / T90")
else:
    print("Too few eligible or distinct milestone vectors for two groups; retain the status comparison.")

,unit_name,threshold,day,status,observation_bracket,baseline_sensitivity,quality_flag
0,Tacloban City,T50,148.0,supported; persistent return after observed nadir,148–148 d,148–356 d,OBS_OK
1,Tacloban City,T80,NaN,not recovered by 363 d,None,≥316 d; conservative crossing absent,OBS_OK
2,Tacloban City,T90,NaN,not recovered by 363 d,None,≥356 d; conservative crossing absent,OBS_OK
3,Palo,T50,104.0,supported; persistent return after observed nadir,104–104 d,16–148 d,OBS_OK
4,Palo,T80,148.0,supported; persistent return after observed nadir,148–148 d,116–284 d,OBS_OK
5,Palo,T90,148.0,supported; persistent return after observed nadir,148–148 d,≥140 d; conservative crossing absent,OBS_OK
6,Guiuan,T50,16.0,supported; persistent return after observed nadir,12–16 d,16–16 d,OBS_OK
7,Guiuan,T80,16.0,supported; persistent return after observed nadir,12–16 d,16–344 d,OBS_OK
8,Guiuan,T90,108.0,supported; persistent return after observed nadir,108–108 d,≥16 d; conservative crossing absent,OBS_OK
9,Alangalang,T50,52.0,supported; persistent return after observed nadir,52–52 d,20–136 d,OBS_OK


,curve_family,milestone_family,milestone_eligible,T50_day,T80_day,T90_day,quality_flag
unit_name,,,,,,,
Tacloban City,Not grouped,Not grouped,False,148,NaN,NaN,OBS_OK
Palo,Not grouped,Family 2,True,104,148.0,148.0,OBS_OK
Guiuan,Not grouped,Family 1,True,16,16.0,108.0,OBS_OK
Alangalang,Not grouped,Family 2,True,52,136.0,140.0,OBS_OK
Ormoc City,Not grouped,Family 2,True,104,164.0,168.0,OBS_OK
Baybay City,Not grouped,Family 1,True,32,44.0,72.0,OBS_OK


## 7. Repeat after removing dim baseline pixels

This is a **sensitivity test**, not a rule that low radiance is wrong. Within each
municipality, remove pixels below its pre-event 25th baseline percentile. Keep the
original daily P95 values, dates, pixel baselines, SC threshold and metric function.
The map shows exactly which example pixels are removed. The six-panel plot compares
the original and brighter-support profiles on dates observed in both versions.

Report how much baseline radiance and how many pixels remain, plus retained dates.
SC is relative to each version's own support, so matching dates alone does not remove
all spatial-support differences. Removing dim pixels also changes the represented
settlements: a changed result flags joint brightness/support sensitivity, not proof
of measurement bias. A stable result does not rule out all bias either.

In [16]:
bright_profiles, sensitivity_rows, bright_masks = [], [], {}
for name in LOCATION_ORDER:
    obj = cache[name]
    b = obj["B"]
    finite_b = b.values[np.isfinite(b.values)]
    cutoff = float(np.quantile(finite_b, DIM_QUANTILE))
    keep = (b >= cutoff) & obj["fixed"]
    bright_masks[name] = keep
    local = rq_cube.sel(x=b.x, y=b.y)
    p, _, _, _, report = build_pixel_matched_profile(
        local, keep, 4, name, "City/municipality")
    bright_profiles.append(p)
    old = poc_profiles.loc[poc_profiles.unit_name == name].set_index("block")
    paired = old[["recovery_pct"]].join(p.set_index("block")[["recovery_pct"]],
        lsuffix="_original", rsuffix="_brighter").loc[0:90].dropna()
    sensitivity_rows.append(dict(unit_name=name, cutoff=cutoff,
        pixels_retained_pct=100*int(keep.sum())/int(obj["fixed"].sum()),
        baseline_radiance_retained_pct=100*float(b.where(keep).sum())/float(b.sum()),
        original_post_composites=int(old.loc[0:90].recovery_pct.count()),
        brighter_post_composites=int(p.loc[p.block.between(0,90)].recovery_pct.count()),
        paired_post_composites=len(paired),
        paired_median_abs_change_pp=(paired.recovery_pct_original-paired.recovery_pct_brighter).abs().median()))
bright_profiles = pd.concat(bright_profiles, ignore_index=True)
sensitivity = pd.DataFrame(sensitivity_rows).set_index("unit_name")
display(sensitivity.round(2))

fig = make_subplots(rows=1, cols=1)
m = xr.where(example["fixed"], xr.where(bright_masks[POC_UNIT], 1., 0.), np.nan)
fig.add_trace(go.Heatmap(x=m.x.values, y=m.y.values, z=m.values, zmin=0, zmax=1,
    colorscale=[[0,"#D55E00"],[.499,"#D55E00"],[.5,"#0072B2"],[1,"#0072B2"]],
    colorbar=dict(tickvals=[0,1], ticktext=["Removed dim support", "Retained brighter support"]),
    hoverongaps=False), row=1, col=1)
add_outline(fig, POC_UNIT)
show(fig, "13_dim_pixel_exclusion_map", f"7 · {POC_UNIT}: which pixels change in the sensitivity run?", 520)

fig = make_subplots(rows=2, cols=3, subplot_titles=LOCATION_ORDER, shared_xaxes=True)
for i, name in enumerate(LOCATION_ORDER):
    row, col = i//3+1, i%3+1
    a = full_series(poc_profiles.loc[poc_profiles.unit_name == name], "recovery_pct")
    b = full_series(bright_profiles.loc[bright_profiles.unit_name == name], "recovery_pct")
    common = a.notna() & b.notna()
    for vals, label, color, dash in ((a, "Original support", "#777777", "solid"),
            (b, "Brighter support", "#0072B2", "dash")):
        fig.add_trace(go.Scatter(x=vals.index*4+1.5, y=vals.where(common), mode="lines",
            name=label, showlegend=(i==0), connectgaps=False,
            line=dict(color=color, dash=dash)), row=row, col=col)
    fig.add_hline(y=100, line_dash="dot", row=row, col=col)
    fig.add_vline(x=0, line_dash="dash", line_color="#0057FF", row=row, col=col)
fig.update_xaxes(title_text="Days since Haiyan")
fig.update_yaxes(title_text="% baseline")
show(fig, "14_baseline_sensitivity_curves", "7 · Original and brighter support, on shared observation dates", 680)

# Equal date support before aggregating: isolate brightness/support sensitivity
# from an accidental change in the dates entering each bin.
paired_dates = poc_profiles[["unit_name","block","recovery_pct"]].merge(
    bright_profiles[["unit_name","block","recovery_pct"]], on=["unit_name","block"],
    suffixes=("_original","_brighter"))
paired_dates.loc[paired_dates[["recovery_pct_original","recovery_pct_brighter"]].isna().any(axis=1),
    ["recovery_pct_original","recovery_pct_brighter"]] = np.nan
common_matrices = []
for variant in ("original", "brighter"):
    f, _ = make_features(paired_dates[["unit_name","block",f"recovery_pct_{variant}"]]
        .rename(columns={f"recovery_pct_{variant}":"recovery_pct"}))
    common_matrices.append(f)
common_units = common_matrices[0].notna().all(axis=1) & common_matrices[1].notna().all(axis=1)
original_common = group_curves(common_matrices[0].loc[common_units])
brighter_common = group_curves(common_matrices[1].loc[common_units])
# Align arbitrary family numbers by maximum membership overlap before comparing.
if original_common.ne("Not grouped").all() and len(original_common) >= 4:
    tab = pd.crosstab(original_common, brighter_common).reindex(
        index=["Family 1","Family 2"], columns=["Family 1","Family 2"], fill_value=0)
    rr, cc = linear_sum_assignment(-tab.to_numpy())
    mapping = {tab.columns[j]:tab.index[i] for i,j in zip(rr,cc)}
    brighter_common = brighter_common.map(mapping).fillna("Not grouped")
assignment_check = pd.DataFrame({"original_same_dates":original_common,
    "brighter_same_dates":brighter_common}).reindex(LOCATION_ORDER)
assignment_check["changed"] = np.where(assignment_check.isna().any(axis=1)
    | assignment_check.eq("Not grouped").any(axis=1), "not compared",
    np.where(assignment_check.original_same_dates == assignment_check.brighter_same_dates, "no", "yes"))
display(assignment_check)

bright_metrics = pd.DataFrame([calculate_recovery_metrics(p)
    for _, p in bright_profiles.groupby("unit_name", observed=True)]).set_index("unit_name")
milestone_sensitivity = []
for name in LOCATION_ORDER:
    for t in (50,80,90):
        milestone_sensitivity.append(dict(unit_name=name, threshold=f"T{t}",
            original_day=metrics.loc[name,f"T{t}_day"],
            brighter_day=bright_metrics.loc[name,f"T{t}_day"],
            original_status=metrics.loc[name,f"T{t}_status"],
            brighter_status=bright_metrics.loc[name,f"T{t}_status"]))
milestone_sensitivity = pd.DataFrame(milestone_sensitivity)
display(milestone_sensitivity)

/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_i

,cutoff,pixels_retained_pct,baseline_radiance_retained_pct,original_post_composites,brighter_post_composites,paired_post_composites,paired_median_abs_change_pp
unit_name,,,,,,,
Tacloban City,2.15,74.83,94.79,70,68,68,0.82
Palo,0.55,74.51,94.48,71,71,71,1.86
Guiuan,0.78,75.86,93.19,65,64,64,2.27
Alangalang,0.27,73.81,88.57,73,71,71,3.84
Ormoc City,0.43,75.00,94.91,74,74,74,1.64
Baybay City,0.33,74.44,90.26,70,70,70,3.21


,original_same_dates,brighter_same_dates,changed
unit_name,,,
Tacloban City,NaN,NaN,not compared
Palo,NaN,NaN,not compared
Guiuan,NaN,NaN,not compared
Alangalang,NaN,NaN,not compared
Ormoc City,Not grouped,Not grouped,not compared
Baybay City,Not grouped,Not grouped,not compared


,unit_name,threshold,original_day,brighter_day,original_status,brighter_status
0,Tacloban City,T50,148.0,148.0,supported; persistent return after observed nadir,supported; persistent return after observed nadir
1,Tacloban City,T80,NaN,NaN,not recovered by 363 d,not recovered by 363 d
2,Tacloban City,T90,NaN,NaN,not recovered by 363 d,not recovered by 363 d
3,Palo,T50,104.0,104.0,supported; persistent return after observed nadir,supported; persistent return after observed nadir
4,Palo,T80,148.0,148.0,supported; persistent return after observed nadir,supported; persistent return after observed nadir
5,Palo,T90,148.0,148.0,supported; persistent return after observed nadir,supported; persistent return after observed nadir
6,Guiuan,T50,16.0,88.0,supported; persistent return after observed nadir,supported; persistent return after observed nadir
7,Guiuan,T80,16.0,88.0,supported; persistent return after observed nadir,supported; persistent return after observed nadir
8,Guiuan,T90,108.0,108.0,supported; persistent return after observed nadir,supported; persistent return after observed nadir
9,Alangalang,T50,52.0,104.0,supported; persistent return after observed nadir,supported; persistent return after observed nadir


## 8. A modest first step toward stable references

Before selecting an unaffected donor, ask how the same places behave before the
event. Use days **−180 to −61** to build an alternative training baseline; hold out
days **−60 to −1** for inspection. Rebuild pixel eligibility from the training period
alone (at least 10 of 30 four-day composites and a positive median). Reuse Notebook 3's
daily RQ observations; do not use its later 60-day baseline mask here.

The panel shows training and held-out matched-baseline ratios. The candidate with
the smallest training MAD is highlighted in the table; the held-out data do not
choose it. Require at least 15 training and 8 held-out municipality composites before
ranking. The held-out median deviation and variability test whether apparent stability
carries forward. These are exploratory thresholds, not validated RQ requirements.

This is a **pre-event stability check**, not an unaffected control or a causal
counterfactual. All six locations were selected as affected comparisons. Independent
hazard, outage and local evidence is needed to establish an uninterrupted donor;
seasonal matching and other years are needed to establish normal behaviour.

In [17]:
reference_rows, reference_curves = [], {}
fig = make_subplots(rows=2, cols=3, subplot_titles=LOCATION_ORDER, shared_yaxes=True)
for i, name in enumerate(LOCATION_ORDER):
    obj = cache[name]
    # Deliberately rebuild from the original RQ cube, not C (which carries the later baseline mask).
    raw = rq_cube.sel(x=obj["B"].x, y=obj["B"].y, date=slice(ANALYSIS_START, PRE_EVENT_END))
    raw = raw.where(obj["support"])
    day_index = pd.DatetimeIndex(raw.date.values).normalize()
    block = np.floor_divide((day_index-EVENT_DATE).days, 4).astype(int)
    pre = raw.assign_coords(block=("date", block)).groupby("block").median("date", skipna=True)
    pre = pre.reindex(block=np.arange(-45,0)).compute()
    train = pre.sel(block=slice(-45,-16))
    training_B = train.median("block", skipna=True)
    training_mask = (train.count("block") >= 10) & (training_B > 0)
    npixels = int(training_mask.sum())
    if npixels == 0:
        reference_rows.append(dict(unit_name=name, train_n=0, test_n=0, eligible=False))
        continue
    valid = pre.notnull() & training_mask
    sc = 100*valid.sum(SPATIAL_DIMS)/npixels
    ratio = 100*pre.where(valid).sum(SPATIAL_DIMS, min_count=1) / training_B.where(valid).sum(SPATIAL_DIMS, min_count=1)
    ratio = ratio.where(sc >= SPATIAL_COMPLETENESS_PCT)
    a = ratio.sel(block=slice(-45,-16)).values
    b = ratio.sel(block=slice(-15,-1)).values
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    row = dict(unit_name=name, training_pixels=npixels, train_n=len(a), test_n=len(b),
        eligible=(len(a)>=15 and len(b)>=8),
        train_mad_pp=np.median(abs(a-np.median(a))) if len(a) else np.nan,
        test_median_deviation_pp=np.median(b)-100 if len(b) else np.nan,
        test_mad_pp=np.median(abs(b-np.median(b))) if len(b) else np.nan)
    reference_rows.append(row)
    reference_curves[name] = ratio.to_series()
    rr, cc = i//3+1, i%3+1
    fig.add_trace(trace(ratio.block.values*4+1.5, ratio.values, name, LOCATION_COLORS[name],
        showlegend=False), row=rr, col=cc)
    fig.add_vrect(x0=-60, x1=-1, fillcolor="#56B4E9", opacity=.12, line_width=0, row=rr, col=cc)
    fig.add_hline(y=100, line_dash="dot", row=rr, col=cc)
fig.update_xaxes(title_text="Days before Haiyan; shaded = held out")
fig.update_yaxes(title_text="% of training baseline")
show(fig, "15_pre_event_holdout", "8 · Does pre-event stability persist into the held-out baseline period?", 650)
references = pd.DataFrame(reference_rows).set_index("unit_name")
references["pre_event_candidate"] = False
valid_refs = references.loc[references.eligible]
if len(valid_refs):
    references.loc[valid_refs.train_mad_pp.idxmin(), "pre_event_candidate"] = True
display(references)

,training_pixels,train_n,test_n,eligible,train_mad_pp,test_median_deviation_pp,test_mad_pp,pre_event_candidate
unit_name,,,,,,,,
Tacloban City,147,18,10,True,11.710884,-3.251953,26.624874,True
Palo,102,17,10,True,18.773720,-12.203636,16.139416,False
Guiuan,29,18,6,False,19.632359,22.944382,38.538490,False
Alangalang,42,18,11,True,25.730087,-4.047569,27.412697,False
Ormoc City,198,18,7,False,21.917702,-3.913864,17.760147,False
Baybay City,90,19,8,True,30.053566,-0.456917,10.431908,False


## What to take to the next meeting

Use the generated maps and examples to describe what you actually observe:

1. **Summarisation:** does the functional central envelope convey something the six
   individual curves do not? Which actual municipality is deepest, and why?
2. **Grouping:** which pairs are close in the distance matrix? Does either two-family
   mean hide important member differences? Are sparse dates driving eligibility?
3. **Timing:** which thresholds were never lost, which returns are supported, and
   which remain unresolved? Do curve and milestone groups answer different questions?
4. **Sensitivity:** do membership, T-status or supported return days change after
   removing dim support? How much represented baseline radiance and territory changed?
5. **References:** does the pre-event candidate remain stable in the held-out period?
   What independent evidence would establish a genuinely uninterrupted reference?

Only after these examples make sense should we extend to all municipalities or
pixel trajectories, compare cluster numbers, assess spatial resampling stability,
test brightness/coverage strata and explore spatial constraints or time warping.

### Method sources and what is borrowed

- [Sun & Genton (2011), Functional Boxplots](https://doi.org/10.1198/jcgs.2011.09224):
  whole-curve depth ranking and central envelopes. This is a general statistical
  method; we are applying it here to NTL recovery curves, not claiming it is an
  established Black Marble recovery procedure.
- [Zheng et al. (2025), Remote Sensing of Environment](https://doi.org/10.1016/j.rse.2025.114645):
  NTL recovery-trajectory clustering provides a relevant precedent. Their BEAST and
  time-weighted dynamic-time-warping workflow is more elaborate. This POC uses
  event-aligned Euclidean grouping so the inputs and distances remain visible.
- [Small (2021), Frontiers in Remote Sensing](https://doi.org/10.3389/frsen.2021.775399):
  NTL spatiotemporal variability motivates inspecting brightness and variability;
  monthly-scale findings do not establish daily stability here.
- [Mu et al. (2025), IEEE TGRS](https://doi.org/10.1109/TGRS.2024.3512549):
  relevant NTL counterfactual work. The pre-event split above is a diagnostic precursor,
  not a replication of a synthetic-control or unaffected-donor estimator.

**Validation of this deliverable:** copied source functions checked against the
uploaded Notebook 3; notebook structure and Python syntax checked; new numeric and
plotting helpers checked separately. Full real-data execution requires your existing
local project datasets. No group assignments or recovery results are pre-filled.

In [18]:
# Small review tables; figures stay interactive in the notebook.
for filename, table in {
    "poc_profiles.csv": poc_profiles,
    "poc_20day_features.csv": features.reset_index(),
    "poc_20day_counts.csv": counts.reset_index(),
    "poc_group_comparison.csv": group_comparison.reset_index(),
    "poc_milestones.csv": milestones,
    "poc_baseline_sensitivity.csv": sensitivity.reset_index(),
    "poc_assignment_sensitivity.csv": assignment_check.reset_index(),
    "poc_milestone_sensitivity.csv": milestone_sensitivity,
    "poc_pre_event_references.csv": references.reset_index(),
}.items():
    table.to_csv(POC_OUT / filename, index=False)
print("Review tables written to", POC_OUT)
print("Interactive figures created:", len(figures))

Review tables written to /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/output/visual_recovery_poc
Interactive figures created: 14
